# The Training Loop

This is the core of all deep learning. Every model — tiny MLP to GPT-4 — trains with this exact same loop.

```
for each epoch:
    for each batch:
        1. forward pass   → compute predictions
        2. compute loss   → how wrong are we?
        3. zero_grad      → clear old gradients
        4. backward       → compute new gradients
        5. optimizer step → update weights
```

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# --- Dataset: y = 2x + 1 (we want the model to learn this) ---
X = torch.randn(100, 1)
y = 2 * X + 1 + 0.1 * torch.randn(100, 1)   # add small noise

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# --- Model ---
model   = nn.Linear(1, 1)   # one input, one output — learns y = Wx + b

# --- Loss ---
loss_fn = nn.MSELoss()

# --- Optimizer ---
# SGD: W = W - lr * gradient  (basic, but foundational)
# Adam: adaptive learning rate per parameter (most commonly used in practice)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print('Initial weight:', model.weight.item())
print('Initial bias:  ', model.bias.item())
print('Should learn:  weight≈2.0, bias≈1.0')

In [ ]:
# --- THE TRAINING LOOP ---

for epoch in range(200):

    model.train()   # set to training mode

    # 1. Forward pass
    y_pred = model(X)

    # 2. Compute loss
    loss = loss_fn(y_pred, y)

    # 3. Zero gradients — MUST do this before backward
    optimizer.zero_grad()

    # 4. Backward pass — compute gradients
    loss.backward()

    # 5. Update weights
    optimizer.step()

    if epoch % 40 == 0:
        print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f} | W: {model.weight.item():.3f} | b: {model.bias.item():.3f}')

print(f'\nFinal weight: {model.weight.item():.3f} (target: 2.0)')
print(f'Final bias:   {model.bias.item():.3f} (target: 1.0)')

## Adding evaluation — train vs val split

Real training always tracks validation loss to detect overfitting.

In [ ]:
torch.manual_seed(42)

# Data
X = torch.randn(200, 1)
y = 3 * X + 2 + 0.1 * torch.randn(200, 1)

# Train/val split — first 160 train, last 40 val
X_train, y_train = X[:160], y[:160]
X_val,   y_val   = X[160:], y[160:]

model     = nn.Linear(1, 1)
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):

    # --- TRAIN ---
    model.train()
    y_pred   = model(X_train)
    loss     = loss_fn(y_pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # --- EVAL ---
    model.eval()
    with torch.no_grad():   # no gradients needed for eval
        val_pred = model(X_val)
        val_loss = loss_fn(val_pred, y_val)

    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | Train loss: {loss.item():.4f} | Val loss: {val_loss.item():.4f}')

## Optimizers — what they do

| Optimizer | How it updates weights | When to use |
|---|---|---|
| SGD | W -= lr * grad | Simple, needs careful lr tuning |
| SGD + momentum | Uses moving average of gradients | Better than vanilla SGD |
| Adam | Adaptive lr per parameter | Default choice for most tasks |
| AdamW | Adam + proper weight decay | Default for LLM training |

Under the hood they all do: **new_weight = old_weight - lr * gradient**
The difference is how they transform the raw gradient before applying it.

In [ ]:
# AdamW — the standard for LLM training
# weight_decay adds L2 regularization: penalizes large weights
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01    # regularization — prevents overfitting
)
print('AdamW created:', optimizer)